In [1]:
import pandas as pd
import numpy as np
import random
import warnings


# Suppress all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
# Backtest trades function
def backtest_trades(price_data, signal_data, tp=None, sl=None, entry_time_offset=None,
                    percentage_change=None, time_limit_minutes=None, ignore_time_interval_before=None, ignore_time_interval_after=None):
    output_data = pd.DataFrame(columns=[
        'Datetime', 'Side', 'Signal Open Price', 'Entry Price', 'TP Price', 'SL Price', 'Result', 'Duration',
        'Execution Latency', 'ROI', 'NAV', 'Ignore Reason'
    ])
    
    initial_margin = 100000
    current_margin = initial_margin
    exit_datetimes = []

    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']
        
        if signal_value == 0:
            continue
        elif signal_value > 0:
            side = 'Buy'
        else:
            side = 'Sell'
        
        adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
        if adjusted_signal_datetime not in price_data.index:
            continue

        signal_open_price = price_data.at[adjusted_signal_datetime, 'Open']
        
        # Ignoring signals based on open trades
        exit_datetimes.sort(key=lambda x: x[0])
        ignore_signal = False
        reason = ''
        if exit_datetimes:
            later_exits = [ed for ed in exit_datetimes if ed[0] > signal_datetime]

            if len(later_exits) >= 3:
                result = 'Ignored'
                reason = 'More than 2 open trades'
                ignore_signal = True
            elif len(later_exits) == 2:
                if later_exits[-1][1] == side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'Two open trades, last one with different side'
                    ignore_signal = True
            elif len(later_exits) == 1:
                if later_exits[-1][1] != side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'One open trade with the same side'
                    ignore_signal = True

        if ignore_signal:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': result,
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': reason
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        # New logic: ignore signals within a specific time interval before and after events
        if 'event_datetimes' in row and pd.notna(row['event_datetimes']):
            event_times = [pd.to_datetime(e.strip()) for e in str(row['event_datetimes']).split(',')]
            ignore_signal = any(event_datetime - pd.Timedelta(minutes=ignore_time_interval_before) <= signal_datetime <= event_datetime + pd.Timedelta(minutes=ignore_time_interval_after)
                                for event_datetime in event_times)
            if ignore_signal:
                new_row = pd.DataFrame([{
                    'Datetime': signal_datetime,
                    'Side': side,
                    'Signal Open Price': signal_open_price,
                    'Entry Price': None,
                    'TP Price': None,
                    'SL Price': None,
                    'Result': 'Ignored',
                    'Duration': '00:00:00',
                    'Execution Latency': '00:00:00',
                    'ROI': 0,
                    'NAV': current_margin,
                    'Ignore Reason': 'Signal around economic event'
                }])
                output_data = pd.concat([output_data, new_row], ignore_index=True)
                continue

        entry_datetime, entry_price, entry_duration = determine_entry(price_data, signal_datetime, percentage_change,
                                                                      side, time_limit_minutes, entry_time_offset)
        if entry_datetime is None:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': 'Not Filled',
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': ''
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue
        
        if side == 'Buy':
            tp_price = entry_price * (1 + tp)
            sl_price = entry_price * (1 - sl)
        else:
            tp_price = entry_price * (1 - tp)
            sl_price = entry_price * (1 + sl)
        
        result, duration_str = check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side)
        exit_datetime = entry_datetime + pd.Timedelta(duration_str)

        if result in [1, -1]:
            exit_datetimes.append((exit_datetime, side))

        # Check for economic data event before the trade exit
        if 'event_datetimes' in row and pd.notna(row['event_datetimes']):
            event_times = [pd.to_datetime(e.strip()) for e in str(row['event_datetimes']).split(',')]
            for event_datetime in event_times:
                if entry_datetime < event_datetime < exit_datetime:
                    exit_datetime = event_datetime - pd.Timedelta(minutes=10)
                    if exit_datetime in price_data.index:
                        exit_price = price_data.at[exit_datetime, 'Open']
                        result = 'ended before data'
                        if (side == 'Buy' and exit_price > entry_price) or (side == 'Sell' and exit_price < entry_price):
                            result += ' with profit'
                            pct_change = (exit_price - entry_price) / entry_price if side == 'Buy' else (entry_price - exit_price) / entry_price
                            current_margin = current_margin * (1 + pct_change)
                        else:
                            result += ' with loss'
                            pct_change = (entry_price - exit_price) / entry_price if side == 'Buy' else (exit_price - entry_price) / entry_price
                            current_margin = current_margin * (1 - pct_change)
                    else:
                        exit_price = price_data.iloc[price_data.index.get_loc(exit_datetime, method='nearest')]['Open']
                        result = 'ended before data with no exact price'
                    break

        if result not in ['ended before data with profit', 'ended before data with loss', 'ended before data with no exact price']:
            if result == 1:
                current_margin = current_margin * (1 + tp)
            elif result == -1:
                current_margin = current_margin * (1 - sl)
        
        roi = ((current_margin - initial_margin) / initial_margin) * 100
        nav = current_margin
        initial_margin = current_margin
        
        new_row = pd.DataFrame([{
            'Datetime': signal_datetime,
            'Side': side,
            'Signal Open Price': signal_open_price,
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
            'Result': result,
            'Duration': duration_str,
            'Execution Latency': format_duration(entry_duration),
            'ROI': roi,
            'NAV': nav,
            'Ignore Reason': ''
        }])
        
        output_data = pd.concat([output_data, new_row], ignore_index=True)
    
    return output_data

# Helper functions
def format_duration(duration):
    seconds = duration.total_seconds()
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}"

def check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side):
    result = 0
    exit_datetime = None
    subsequent_prices = price_data.loc[entry_datetime:]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy':
            if price_row['High'] >= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['Low'] <= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
        else:
            if price_row['Low'] <= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['High'] >= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
    
    if exit_datetime:
        duration = exit_datetime - entry_datetime
        duration_str = format_duration(duration)
    else:
        duration_str = '00:00:00'
        
    return result, duration_str

def determine_entry(price_data, signal_datetime, percentage_change, side, time_limit_minutes, entry_time_offset):
    adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
    if adjusted_signal_datetime not in price_data.index:
        return None, None, None
    
    adjusted_open_price = price_data.at[adjusted_signal_datetime, 'Open']
    percentage_change_price = adjusted_open_price * (1 - percentage_change) if side == 'Buy' else adjusted_open_price * (1 + percentage_change)
    
    time_limit = adjusted_signal_datetime + pd.Timedelta(minutes=time_limit_minutes)
    subsequent_prices = price_data.loc[adjusted_signal_datetime:time_limit]
    
    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy' and price_row['Low'] <= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration
        elif side == 'Sell' and price_row['High'] >= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration

    return None, None, None

In [3]:
# Load the new datasets
price_data = pd.read_csv('E:\\Signal Backtesting\\Input\\Price_2024.csv', 
                         parse_dates=['Datetime'], index_col='Datetime')

signal_data = pd.read_csv('E:\Signal Backtesting\Input\\Updated_2024_Signal_Dataset_with_events.csv', 
                          parse_dates=['Datetime'])

In [4]:
# Set a random seed for reproducibility
np.random.seed(42)
random.seed(42)

# Define the parameter ranges
tp_values = np.arange(0.009, 0.014, 0.001)
sl_values = np.arange(0.009, 0.014, 0.001)
entry_time_offset_values = np.arange(60, 200, 10)
percentage_change_values = np.arange(0.0001, 0.0008, 0.0001)
ignore_time_interval_before = np.arange(0, 1080, 60)

# Evaluation function
def evaluate(individual):
    tp, sl, entry_time_offset, percentage_change, ignore_time_interval_before = individual
    
    # Filter data for the specific month
    month_price_data = price_data[price_data.index.month == specific_month]
    month_signal_data = signal_data[signal_data['Datetime'].dt.month == specific_month]

    # Ensure that the entry_time_offset does not exceed the length of the month_price_data
    entry_time_offset = min(entry_time_offset, len(month_price_data) - 1)
    
    # Run backtest with given parameters
    result = backtest_trades(
        month_price_data, month_signal_data, tp=tp, sl=sl, 
        entry_time_offset=entry_time_offset, 
        percentage_change=percentage_change, time_limit_minutes=120, ignore_time_interval_before=ignore_time_interval_before, ignore_time_interval_after=0
    )
    
    # Calculate the final NAV
    final_nav = result['NAV'].iloc[-1]
    
    # Calculate ROI
    roi = ((final_nav - 100000) / 100000) * 100
    
    return roi

# Randomly initialize an individual
def create_individual():
    return [
        np.random.choice(tp_values),
        np.random.choice(sl_values),
        np.random.choice(entry_time_offset_values),
        np.random.choice(percentage_change_values),
        np.random.choice(ignore_time_interval_before)
    ]

# Mutate an individual
def mutate(individual):
    index = random.randint(0, len(individual) - 1)
    if index == 0:
        individual[index] = np.random.choice(tp_values)
    elif index == 1:
        individual[index] = np.random.choice(sl_values)
    elif index == 2:
        individual[index] = np.random.choice(entry_time_offset_values)
    elif index == 3:
        individual[index] = np.random.choice(percentage_change_values)
    elif index == 4:
        individual[index] = np.random.choice(ignore_time_interval_before)
    return individual

# Simulated Annealing algorithm
def simulated_annealing():
    current_individual = create_individual()
    current_fitness = evaluate(current_individual)
    best_individual = list(current_individual)
    best_fitness = current_fitness
    
    initial_temperature = 1.0
    final_temperature = 0.001
    alpha = 0.99
    temperature = initial_temperature
    
    while temperature > final_temperature:
        new_individual = mutate(list(current_individual))
        new_fitness = evaluate(new_individual)
        
        if new_fitness > current_fitness or random.uniform(0, 1) < np.exp((new_fitness - current_fitness) / temperature):
            current_individual = new_individual
            current_fitness = new_fitness
        
        if current_fitness > best_fitness:
            best_individual = list(current_individual)
            best_fitness = current_fitness
        
        temperature *= alpha
    
    best_tp, best_sl, best_entry_time_offset, best_percentage_change, best_ignore_time_interval_before = best_individual
    optimized_roi = best_fitness
    
    print(f"Month: {specific_month}")
    print(f"Best Take Profit: {best_tp}")
    print(f"Best Stop Loss: {best_sl}")
    print(f"Best Entry Time Offset: {best_entry_time_offset}")
    print(f"Best Percentage Change: {best_percentage_change}")
    print(f"Best ignore time offset: {best_ignore_time_interval_before}")
    print(f"Optimized ROI: {optimized_roi:.4f}")

# Function to optimize for each month
def optimize_for_month(month):
    global specific_month
    specific_month = month
    simulated_annealing()


In [ ]:
# Example: Optimize for January
optimize_for_month(6)

In [6]:
# Example: Optimize for January
optimize_for_month(5)

Month: 5
Best Take Profit: 0.012999999999999996
Best Stop Loss: 0.013999999999999995
Best Entry Time Offset: 60
Best Percentage Change: 0.0006000000000000001
Optimized ROI: 12.8493


In [7]:
# Example: Optimize for January
optimize_for_month(4)

Month: 4
Best Take Profit: 0.011999999999999997
Best Stop Loss: 0.011999999999999997
Best Entry Time Offset: 70
Best Percentage Change: 0.0007000000000000001
Optimized ROI: 19.4902


In [8]:
# Example: Optimize for January
optimize_for_month(3)

Month: 3
Best Take Profit: 0.009
Best Stop Loss: 0.013999999999999995
Best Entry Time Offset: 110
Best Percentage Change: 0.0006000000000000001
Optimized ROI: 10.7652


In [9]:
# Example: Optimize for January
optimize_for_month(2)

Month: 2
Best Take Profit: 0.012999999999999996
Best Stop Loss: 0.009999999999999998
Best Entry Time Offset: 70
Best Percentage Change: 0.0005
Optimized ROI: 8.5213


In [10]:
# Example: Optimize for January
optimize_for_month(1)

Month: 1
Best Take Profit: 0.009
Best Stop Loss: 0.013999999999999995
Best Entry Time Offset: 140
Best Percentage Change: 0.0004
Optimized ROI: 18.8553
